# 01b · EverydayScents: R1 raw features

**Purpose.** Downsample each sensor curve to T' = 70 and flatten (R1). Fit the 4 heads on both tasks with StratifiedGroupKFold(5), groups = session, 3 seeds.

**Inputs**
- EverydayScents `X_signals.npy` (144, 11, 210) + `meta.csv`

**Outputs**
- F_raw: (144, 11, 70) → (144, 770)
- `results/01b_es_raw.csv`: one row per seed, fold, task, rep, head, metrics

In [ ]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import TASKS, load_baseline_level, load_processed, precomputed, run_cv, save_results, summarize
from src.features.raw import raw_features
DATASET = "es"
REP = "R1_raw"
NAME = "01b_es_raw"          # results/01b_es_raw.csv

X, meta, info = load_processed(DATASET)    # X: (N, C, T) baseline corrected
print(X.shape, meta["y_multi"].value_counts().sort_index().tolist())

## R1: downsample every curve to T' = 70 points and flatten
Nothing is fitted, so the features are computed once for all samples.

In [ ]:
T_PRIME = 70
F = raw_features(X, T_PRIME)               # (N, C * T') = (144, 770)
assert F.shape == (144, 770)
F.shape

## Cross validation: 4 heads × 5 folds × 3 seeds, both tasks
Everything that is fitted (scaler, heads, hyperparameters) sees the training fold only.

In [ ]:
scores, preds = [], []
for task in TASKS:                                   # "multi", then "binary"
    s, p = run_cv(F, meta, DATASET, task, REP, precomputed)
    scores.append(s)
    preds.append(p)
scores, preds = pd.concat(scores, ignore_index=True), pd.concat(preds, ignore_index=True)
save_results(scores, preds, NAME)
print("saved results/" + NAME + ".csv")

## Results: mean ± std over 3 seeds × 5 folds

In [ ]:
print("macro F1"); display(summarize(scores))
print("balanced accuracy"); display(summarize(scores, "balanced_acc"))

In [ ]:
print("ROC AUC, binary task"); summarize(scores[scores["task"] == "binary"], "roc_auc")